# Stage 5: Retrieval and Cross-Encoder Reranking

In this stage, we'll learn how to:
1. Load vector stores created in Stage 3
2. Perform initial retrieval (top-k=10) from both stores using keywords
3. Apply cross-encoder reranking to improve relevance
4. Select top-2 documents from each store for code generation

Key Concept: Two-stage retrieval:
- Stage 1 (Dual-Encoder): Fast similarity search over all documents
- Stage 2 (Cross-Encoder): Accurate relevance scoring of top candidates

Cross-encoders jointly encode query+document, capturing fine-grained
semantic relationships that dual-encoders miss.

Output: Top-2 relevant docs/code examples per problem

In [1]:
import os
import pickle
from typing import List
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

from config import (
    VECTORSTORE_DOC_PATH,
    VECTORSTORE_CODE_PATH,
    EMBEDDING_MODEL,
    CROSS_ENCODER_MODEL,
    RETRIEVAL_TOP_K,
    RERANK_TOP_N_DOC,
    RERANK_TOP_N_CODE,
    NUM_RETRIEVAL_EXAMPLES,
    VERBOSE
)

## SECTION 1: LOAD VECTOR STORES

In [2]:
def load_faiss_store(save_path: str, embedding_model_name: str) -> FAISS:
    """
    Load a previously saved FAISS vector store from disk.

    Args:
        save_path: Directory containing index.faiss and index.pkl
        embedding_model_name: Must match the model used during creation

    Returns:
        FAISS vector store instance
    """
    if not os.path.exists(save_path):
        raise FileNotFoundError(f"FAISS store not found: {save_path}")

    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model_name,
        model_kwargs={'trust_remote_code': True}
    )

    faiss_store = FAISS.load_local(
        save_path,
        embeddings,
        allow_dangerous_deserialization=True
    )

    return faiss_store


print("Loading vector stores...")
doc_store = load_faiss_store(VECTORSTORE_DOC_PATH, EMBEDDING_MODEL)
code_store = load_faiss_store(VECTORSTORE_CODE_PATH, EMBEDDING_MODEL)
print(f"   Documentation store loaded")
print(f"   Code examples store loaded\n")

Loading vector stores...
   Documentation store loaded
   Code examples store loaded



## SECTION 2: CROSS-ENCODER RERANKING

In [3]:
print(f"Loading cross-encoder model: {CROSS_ENCODER_MODEL}")
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL)
print(f"   Cross-encoder loaded\n")


def rerank_with_cross_encoder(query: str, docs: List[Document], cross_encoder) -> List[Document]:
    """
    Rerank documents by semantic relevance using cross-encoder.

    How it works:
    1. Create (query, doc) pairs for each candidate document
    2. Cross-encoder scores each pair (0-1, higher = more relevant)
    3. Sort documents by score in descending order

    Args:
        query: Search query (keywords)
        docs: Candidate documents from initial retrieval
        cross_encoder: CrossEncoder model instance

    Returns:
        Documents sorted by relevance (best first)

    Why Cross-Encoders Work Better:
        Dual-encoders (used in FAISS) encode query and doc separately,
        then compute similarity (e.g., cosine). They can't capture
        interactions between query terms and doc terms.

        Cross-encoders encode query+doc together, allowing attention
        mechanisms to find subtle matches. Much more accurate but slower,
        which is why we only use it on top-k candidates.

    Example:
        Query: "minimize transportation costs binary variables"
        Doc 1: "... transportation model ... binary decision variables ..."
        Doc 2: "... minimize costs ... continuous variables ..."

        Dual-encoder might rank Doc 2 higher (more keyword overlap),
        but cross-encoder understands Doc 1 is better (has BOTH
        transportation AND binary variables).
    """
    # Build input pairs
    input_pairs = [(query, doc.page_content) for doc in docs]

    # Get relevance scores
    scores = cross_encoder.predict(input_pairs)

    # Zip docs with scores and sort
    doc_score_pairs = list(zip(docs, scores))
    doc_score_pairs.sort(key=lambda x: x[1], reverse=True)

    # Return sorted documents
    sorted_docs = [pair[0] for pair in doc_score_pairs]

    return sorted_docs, scores


def retrieve_with_rerank(query: str, vectorstore: FAISS, cross_encoder,
                         top_k: int = 10, to_ret: int = 2) -> tuple:
    """
    Combined retrieval + reranking pipeline.

    Args:
        query: Search keywords
        vectorstore: FAISS store (doc or code)
        cross_encoder: CrossEncoder model
        top_k: Initial retrieval count
        to_ret: Final count after reranking

    Returns:
        Tuple of (top_docs, all_scores)
    """
    # Stage 1: Initial vector search
    initial_docs = vectorstore.similarity_search(query, k=top_k)

    # Stage 2: Rerank with cross-encoder
    reranked_docs, scores = rerank_with_cross_encoder(query, initial_docs, cross_encoder)

    # Return top-N after reranking
    return reranked_docs[:to_ret], scores

Loading cross-encoder model: cross-encoder/ms-marco-MiniLM-L-6-v2
   Cross-encoder loaded



## SECTION 3: EXECUTE PIPELINE

In [4]:
print("=" * 80)
print("STAGE 5: RETRIEVAL AND CROSS-ENCODER RERANKING")
print("=" * 80)

# Load keywords from Stage 4
print(f"\nLoading keywords from Stage 4...")
with open('tutorial_keywords.pkl', 'rb') as f:
    keyword_results = pickle.load(f)

print(f"   Loaded keywords for {len(keyword_results)} problems\n")

# Store retrieval results
retrieval_results = []

# Process each problem
print(f"Retrieving and reranking for {len(keyword_results)} problems...\n")

for result in keyword_results:
    problem_id = result['problem_id']
    keywords = result['keywords']

    # Retrieve from documentation store
    doc_retrievals, doc_scores = retrieve_with_rerank(
        keywords, doc_store, cross_encoder,
        top_k=RETRIEVAL_TOP_K,
        to_ret=RERANK_TOP_N_DOC
    )

    # Retrieve from code examples store
    code_retrievals, code_scores = retrieve_with_rerank(
        keywords, code_store, cross_encoder,
        top_k=RETRIEVAL_TOP_K,
        to_ret=RERANK_TOP_N_CODE
    )

    retrieval_results.append({
        'problem_id': problem_id,
        'question': result['question'],
        'keywords': keywords,
        'doc_retrievals': doc_retrievals,
        'doc_scores': doc_scores,
        'code_retrievals': code_retrievals,
        'code_scores': code_scores,
        'expected_objective': result['expected_objective']
    })

    if VERBOSE:
        print(f"   Problem #{problem_id}: Retrieved {len(doc_retrievals)} docs + {len(code_retrievals)} code examples")

STAGE 5: RETRIEVAL AND CROSS-ENCODER RERANKING

Loading keywords from Stage 4...
   Loaded keywords for 2 problems

Retrieving and reranking for 2 problems...

   Problem #0: Retrieved 2 docs + 2 code examples
   Problem #1: Retrieved 2 docs + 2 code examples


In [5]:
# ========================================================================
# DEMONSTRATION: Show Retrieval Results
# ========================================================================

print(f"\n{'=' * 80}")
print("DEMONSTRATION: Retrieval and Reranking Results")
print(f"{'=' * 80}\n")

for result in retrieval_results[:NUM_RETRIEVAL_EXAMPLES]:
    print(f"{'─' * 80}")
    print(f"Problem #{result['problem_id']}")
    print(f"{'─' * 80}\n")

    # Show problem and keywords
    problem_text = result['question']
    if len(problem_text) > 250:
        problem_text = problem_text[:250] + "..."

    print("Problem:")
    print(f"   {problem_text}\n")

    print(f"Keywords: {result['keywords']}\n")

    # Show documentation retrievals
    print(f"Top-{RERANK_TOP_N_DOC} Documentation Retrievals:")
    for i, doc in enumerate(result['doc_retrievals'], 1):
        score = result['doc_scores'][i-1] if i-1 < len(result['doc_scores']) else 0
        print(f"\n  {i}. {doc.metadata['heading_title']}")
        print(f"     Relevance Score: {score:.4f}")
        print(f"     Content: {doc.page_content[:200]}...")

    # Show code retrievals
    print(f"\nTop-{RERANK_TOP_N_CODE} Code Example Retrievals:")
    for i, doc in enumerate(result['code_retrievals'], 1):
        score = result['code_scores'][i-1] if i-1 < len(result['code_scores']) else 0
        print(f"\n  {i}. {doc.metadata['heading_title']}")
        print(f"     Relevance Score: {score:.4f}")
        print(f"     Summary: {doc.metadata['summary'][:150]}...")
        print(f"     Keywords: {doc.metadata['keywords']}")

    print("\n")

# Save results for Stage 6
print(f"\nSaving retrieval results for Stage 6...")
with open('tutorial_retrievals.pkl', 'wb') as f:
    pickle.dump(retrieval_results, f)

print(f"\n{'=' * 80}")
print("✅ STAGE 5 COMPLETE")


DEMONSTRATION: Retrieval and Reranking Results

────────────────────────────────────────────────────────────────────────────────
Problem #0
────────────────────────────────────────────────────────────────────────────────

Problem:
   A fishery wants to transport their catch. They can either use local sled dogs or trucks. Local sled dogs can take 100 fish per trip while trucks can take 300 fish per trip. The cost per trip for sled dogs is $50 while the cost per trip for a truck is...

Keywords: Fishery Transportation, Maximize Fish, Budget Constraint, Trip Constraints, Continuous Variables, Gurobi Model, setObjective

Top-2 Documentation Retrievals:

  1. General Constraints
     Relevance Score: -1.1021
     Content: 1.2.4 General Constraints Gurobi includes an additional set of higher-level constraints, which we collectively refer to as general constraints. Those constraints allow you to directly model complex re...

  2. Constraints
     Relevance Score: -0.0193
     Content: 1.2 Co